In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

# Load API key securely
load_dotenv(dotenv_path=Path("../.env"), override=True)

# Verify
print("API Key loaded:", os.getenv("GROQ_API_KEY") is not None)

API Key loaded: True


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load document — LangChain way
loader = TextLoader("../data/techmart_policy.txt", encoding="utf-8")
documents = loader.load()

print(f"Document loaded!")
print(f"Total characters: {len(documents[0].page_content)}")

# Chunk document — LangChain way
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)
print(f"\nTotal chunks created: {len(chunks)}")
print(f"\nFirst chunk preview:")
print(chunks[0].page_content)

Document loaded!
Total characters: 3149

Total chunks created: 11

First chunk preview:
## About TechMart

TechMart was founded in **2010** and has grown into a trusted retail company with over **500 employees across India**. We are committed to providing high-quality products and excellent customer service nationwide.

---

## 1. Shipping & Delivery Policy


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Create embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vector store and store chunks in one line!
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store created!")
print(f"Total chunks stored: {vectorstore._collection.count()}")

C:\Users\heman\AppData\Local\Temp\ipykernel_40480\3551026633.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created!
Total chunks stored: 11


In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Initialize Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile"
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Manual conversation history — we know this works!
chat_history = []

print("LLM and retriever ready!")
print("LLM: LLaMA 3.3 70B via Groq")
print("Retriever: ChromaDB with 11 chunks")

LLM and retriever ready!
LLM: LLaMA 3.3 70B via Groq
Retriever: ChromaDB with 11 chunks


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("\n--- TechMart AI Chatbot (LangChain Version) ---")
print("Type 'quit' to exit\n")

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "quit":
        print("Thank you for contacting TechMart. Goodbye!")
        break
    
    # Retrieve relevant chunks
    relevant_docs = retriever.invoke(user_input)
    context = "\n".join([doc.page_content for doc in relevant_docs])
    
    # Build messages with history
    messages = [
        ("system", """You are TechMart's helpful customer support assistant.
Use ONLY the context below to answer questions.
If answer not in context, say 'I don't have that information.'

CONTEXT:
{context}"""),
    ]
    
    # Add conversation history
    for msg in chat_history:
        messages.append(msg)
    
    # Add current question
    messages.append(("human", user_input))
    
    # Create and run prompt
    prompt = ChatPromptTemplate.from_messages(messages)
    chain = prompt | llm | StrOutputParser()
    
    response = chain.invoke({"context": context})
    
    # Save to history
    chat_history.append(("human", user_input))
    chat_history.append(("assistant", response))
    
    print(f"\nTechMart Bot: {response}\n")
    print(f"[Memory: {len(chat_history)} messages in history]")


--- TechMart AI Chatbot (LangChain Version) ---
Type 'quit' to exit



AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}